# Bearing fault diagnosis — Colab run (all four operating conditions)

Self-contained: no project files needed. Run the cells in order.

Downloads each Paderborn bearing once (~160 MB, 29 bearings, ~4.6 GB in total), and for **all four operating
conditions** computes the per-window feature table with both the fixed 2–10 kHz demodulation band and an adaptive
band chosen by spectral kurtosis. Progress is saved to Google Drive after every bearing, so if the session drops,
just run the cells again and it continues where it stopped.

Output (in Drive): `features_N15_M07_F10.csv`, `features_N09_M07_F10.csv`, `features_N15_M01_F10.csv`,
`features_N15_M07_F04.csv`. Copy these four files into the project's `data/features/` folder.

Data licence: CC BY-NC 4.0 — cite Lessmeier et al. (2016) and the Paderborn KAt-DataCenter.

## 1. Setup and settings

In [ ]:
!which unrar > /dev/null || apt-get -qq install -y unrar > /dev/null

import shutil
import subprocess
import tempfile
import urllib.request
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.io import loadmat
from scipy.signal import butter, hilbert, sosfiltfilt, stft
from scipy.stats import kurtosis, skew

OUT_DIR = Path("/content/drive/MyDrive/load_bearing_research_paper/data/features")  # kept in Drive
PARTIAL_DIR = OUT_DIR / "partial"                                                    # per-bearing progress

CONDITIONS = {"N15_M07_F10": 1500, "N09_M07_F10": 900, "N15_M01_F10": 1500, "N15_M07_F04": 1500}  # rpm
FS = 64_000                    # vibration sampling rate (Hz)
RECORDINGS = 20                # 4-second recordings per bearing and condition
WINDOW = int(1.0 * FS)         # 1-second windows
BASE_URL = "https://groups.uni-paderborn.de/kat/BearingDataCenter/"

## 2. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
PARTIAL_DIR.mkdir(parents=True, exist_ok=True)

## 3. Bearings and fault frequencies

In [ ]:
HEALTHY, INNER, OUTER = "healthy", "inner_race", "outer_race"

BEARINGS = pd.DataFrame(
    [(c, HEALTHY, "none") for c in ["K001", "K002", "K003", "K004", "K005", "K006"]]
    + [(c, OUTER, "artificial") for c in ["KA01", "KA03", "KA05", "KA06", "KA07", "KA08", "KA09"]]
    + [(c, INNER, "artificial") for c in ["KI01", "KI03", "KI05", "KI07", "KI08"]]
    + [(c, OUTER, "real") for c in ["KA04", "KA15", "KA16", "KA22", "KA30"]]
    + [(c, INNER, "real") for c in ["KI04", "KI14", "KI16", "KI17", "KI18", "KI21"]],
    columns=["bearing", "label", "origin"],
)

# Bearing 6203: 8 balls, ball diameter 6.75 mm, pitch diameter 28.55 mm
N_BALLS, RATIO = 8, 6.75 / 28.55


def fault_freqs(rpm):
    shaft_hz = rpm / 60
    return {"bpfo": N_BALLS / 2 * (1 - RATIO) * shaft_hz, "bpfi": N_BALLS / 2 * (1 + RATIO) * shaft_hz}


for cond, rpm in CONDITIONS.items():
    f = fault_freqs(rpm)
    print(f"{cond}: BPFO {f['bpfo']:.1f} Hz, BPFI {f['bpfi']:.1f} Hz")
print(len(BEARINGS), "bearings")

## 4. Loading recordings

In [ ]:
def channel_name(ch):
    name = ch["Name"]
    while isinstance(name, np.ndarray):
        name = name.flat[0]
    return str(name)


def load_vibration(path, channel="vibration_1"):
    record = loadmat(path)[Path(path).stem][0, 0]
    for ch in record["Y"].ravel():
        if channel_name(ch) == channel:
            return np.asarray(ch["Data"], dtype=float).ravel()
    raise KeyError(f"{channel} not in {[channel_name(c) for c in record['Y'].ravel()]}")

## 5. Features

Identical to the project code (`src/bearing_uq/features/`): time-domain statistics; fault-frequency peaks in the
envelope spectrum with the fixed 2–10 kHz band; and the same peaks with a 4 kHz band centred where the spectral
kurtosis is largest (suffix `_sk`).

In [ ]:
def time_features(x):
    rms = float(np.sqrt(np.mean(x**2)))
    peak = float(np.max(np.abs(x)))
    return {"rms": rms, "kurtosis": float(kurtosis(x, fisher=False)), "skewness": float(skew(x)),
            "crest_factor": peak / rms if rms > 0 else 0.0, "peak_to_peak": float(np.ptp(x))}


def envelope_spectrum(x, band=(2_000.0, 10_000.0)):
    sos = butter(4, list(band), btype="bandpass", fs=FS, output="sos")
    env = np.abs(hilbert(sosfiltfilt(sos, np.asarray(x, dtype=float))))
    env -= env.mean()
    return np.fft.rfftfreq(len(env), d=1 / FS), np.abs(np.fft.rfft(env)) / len(env)


def fault_band_features(freqs, amps, ffreqs, harmonics=2, tol=0.02):
    ref = (freqs >= 10.0) & (freqs <= 500.0)
    floor = float(np.median(amps[ref])) if ref.any() else 0.0
    floor = floor if floor > 0 else 1e-12
    out = {}
    for name, f0 in ffreqs.items():
        for k in range(1, harmonics + 1):
            mask = (freqs >= k * f0 * (1 - tol)) & (freqs <= k * f0 * (1 + tol))
            out[f"{name}_h{k}"] = (float(amps[mask].max()) if mask.any() else 0.0) / floor
    return out


def sk_band(x, bandwidth=4_000.0, f_min=1_000.0, f_max=30_000.0, nperseg=256):
    f, _, z = stft(np.asarray(x, dtype=float), fs=FS, nperseg=nperseg, noverlap=nperseg // 2)
    power = np.abs(z) ** 2
    mean_power = power.mean(axis=1)
    sk = (power ** 2).mean(axis=1) / np.where(mean_power > 0, mean_power ** 2, np.inf) - 2
    width = max(1, int(round(bandwidth / (f[1] - f[0]))))
    smoothed = np.convolve(sk, np.ones(width) / width, mode="same")
    allowed = (f >= f_min + bandwidth / 2) & (f <= f_max - bandwidth / 2)
    centre = float(f[allowed][np.argmax(smoothed[allowed])])
    return centre - bandwidth / 2, centre + bandwidth / 2


def window_features(x, ffreqs):
    freqs, amps = envelope_spectrum(x)
    out = {**time_features(x), **fault_band_features(freqs, amps, ffreqs)}
    band = sk_band(x)
    freqs_sk, amps_sk = envelope_spectrum(x, band)
    out.update({f"{k}_sk": v for k, v in fault_band_features(freqs_sk, amps_sk, ffreqs).items()})
    out["sk_band_centre_hz"] = (band[0] + band[1]) / 2
    return out

## 6. Download, extract and compute features (about 20–40 minutes)

For each bearing: download the archive, extract all four conditions, compute features, save one partial file per
condition to Drive, delete the raw files. Bearings already finished are skipped.

In [ ]:
PARTIAL_DIR.mkdir(parents=True, exist_ok=True)


def partial_path(cond, code):
    return PARTIAL_DIR / f"{cond}_{code}.csv"


def process_bearing(b):
    with tempfile.TemporaryDirectory() as tmp:
        archive = Path(tmp) / f"{b.bearing}.rar"
        with urllib.request.urlopen(BASE_URL + f"{b.bearing}.rar") as r, open(archive, "wb") as f:
            shutil.copyfileobj(r, f, length=1 << 20)
        out = Path(tmp) / "x"
        out.mkdir()
        subprocess.run(["unrar", "x", "-o+", "-inul", str(archive), f"{out}/"], check=True)
        archive.unlink()
        for cond, rpm in CONDITIONS.items():
            ffreqs, rows = fault_freqs(rpm), []
            for n in range(1, RECORDINGS + 1):
                hits = list(out.rglob(f"{cond}_{b.bearing}_{n}.mat"))
                if not hits:
                    print(f"  missing {cond}_{b.bearing}_{n}.mat")
                    continue
                try:
                    signal = load_vibration(hits[0])
                except Exception as e:
                    print(f"  skipped {hits[0].name}: {e}")
                    continue
                for w in range(len(signal) // WINDOW):
                    rows.append({"bearing": b.bearing, "label": b.label, "origin": b.origin, "recording": n,
                                 "window": w, **window_features(signal[w * WINDOW:(w + 1) * WINDOW], ffreqs)})
            pd.DataFrame(rows).to_csv(partial_path(cond, b.bearing), index=False)


for b in BEARINGS.itertuples():
    if all(partial_path(c, b.bearing).exists() for c in CONDITIONS):
        print(f"{b.bearing}: already done")
        continue
    try:
        process_bearing(b)
        print(f"{b.bearing}: done")
    except Exception as e:
        print(f"{b.bearing}: FAILED - {e}")

missing = [(c, b) for b in BEARINGS.bearing for c in CONDITIONS if not partial_path(c, b).exists()]
print("Missing partial files:", missing or "none")

## 7. Combine into one feature table per condition

In [ ]:
for cond in CONDITIONS:
    parts = [pd.read_csv(partial_path(cond, code)) for code in BEARINGS.bearing if partial_path(cond, code).exists()]
    if not parts:
        print(f"{cond}: no bearings processed yet - run step 6 first")
        continue
    table = pd.concat(parts, ignore_index=True)
    table.to_csv(OUT_DIR / f"features_{cond}.csv", index=False)
    print(f"{cond}: {len(table)} windows, {table.bearing.nunique()} bearings -> features_{cond}.csv")

## 8. Sanity check

Median fault-frequency features per group and condition, for the fixed band and the adaptive band. Outer-race
bearings should show higher `bpfo_h1`, inner-race bearings higher `bpfi_h1`.

In [ ]:
for cond in CONDITIONS:
    t = pd.read_csv(OUT_DIR / f"features_{cond}.csv")
    print(f"\n=== {cond} ===")
    print(t.groupby(["origin", "label"])[["kurtosis", "bpfo_h1", "bpfi_h1", "bpfo_h1_sk", "bpfi_h1_sk"]].median().round(2))